In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers

In [ ]:
import torch
import math
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorWithPadding, set_seed
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
import re

In [ ]:
SEED = 42
BATCH_SIZE = 8
MAX_LENGTH = 512
STRIDE = 128
MLM_PROBABILITY = 0.15

In [ ]:
MODELS = [
    "dbmdz/bert-base-italian-xxl-cased",
    "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]
DATASET_PATH = "/content/drive/My Drive/italian_poems/italian_poems_test_rhymes.json"

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fix_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

In [ ]:
for model_id in MODELS:
    print(f"\nEvaluating Head Specificity: {model_id}")
    fix_seed(SEED)

    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if not tokenizer.is_fast:
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    model = AutoModelForMaskedLM.from_pretrained(model_id, output_attentions=True).to(device)
    model.eval()

    # Filter only by length
    def filter_long_samples(examples):
        tokenized = tokenizer(examples["text"], truncation=False, padding=False)
        return [len(ids) <= MAX_LENGTH for ids in tokenized["input_ids"]]

    filtered_dataset = dataset.filter(
        filter_long_samples,
        batched=True,
        desc=f"Filtering > {MAX_LENGTH} tokens"
    )

    # Tokenization and tagging
    def process_data(examples):
        tokenized = tokenizer(
            examples["text"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            return_offsets_mapping=True,
            return_special_tokens_mask=True
        )

        new_input_ids = []
        new_labels = []
        mask_group_ids = []
        target_rhyme_mask = []
        is_rhyming_sample = []

        for i, text in enumerate(examples["text"]):
            raw_tags = examples["rhyme_tags"][i]
            input_ids = tokenized["input_ids"][i]
            offsets = tokenized["offset_mapping"][i]
            special_mask = tokenized["special_tokens_mask"][i]

            labels = [-100] * len(input_ids)
            current_mask_group = [0] * len(input_ids)
            current_target_rhyme = [0] * len(input_ids)

            # Robust line extraction
            line_spans = []
            last_pos = 0
            for match in re.finditer(r'\n', text):
                line_spans.append((last_pos, match.start()))
                last_pos = match.end()
            if last_pos < len(text):
                line_spans.append((last_pos, len(text)))

            # Filter empty lines
            valid_lines = []
            for start, end in line_spans:
                if text[start:end].strip():
                    valid_lines.append((start, end))

            # Determine target tag
            target_tag = None
            has_rhyme = 0
            if len(valid_lines) > 0 and len(raw_tags) == len(valid_lines):
                target_tag = raw_tags[-1]
                if target_tag is not None and raw_tags.count(target_tag) >= 2:
                    has_rhyme = 1
            is_rhyming_sample.append(has_rhyme)

            # Find rhyme targets
            for j, (start_char, end_char) in enumerate(valid_lines[:-1]):
                line_text = text[start_char:end_char]
                current_line_tag = raw_tags[j]

                matches = list(re.finditer(r"\b\w+\b", line_text))
                if matches:
                    last_match = matches[-1]
                    word_start = start_char + last_match.start()
                    word_end = start_char + last_match.end()

                    for idx, (tok_start, tok_end) in enumerate(offsets):
                        if special_mask[idx] or tok_start == tok_end: continue
                        if tok_start < word_end and tok_end > word_start:
                            if target_tag is not None and current_line_tag == target_tag:
                                current_target_rhyme[idx] = 1

            # Masking of last line
            if len(valid_lines) > 0:
                start, end = valid_lines[-1]
                matches = list(re.finditer(r"\b\w+\b", text[start:end]))
                masked_at_least_one = False
                if matches:
                    last_match = matches[-1]
                    word_start = start + last_match.start()
                    word_end = start + last_match.end()
                    for idx, (tok_start, tok_end) in enumerate(offsets):
                        if special_mask[idx] or tok_start == tok_end: continue
                        if tok_start < word_end and tok_end > word_start:
                            labels[idx] = input_ids[idx]
                            input_ids[idx] = tokenizer.mask_token_id
                            current_mask_group[idx] = 1
                            masked_at_least_one = True

                if not masked_at_least_one: # Fallback
                    for idx in range(len(input_ids)-1, -1, -1):
                        if input_ids[idx] != tokenizer.pad_token_id and not special_mask[idx]:
                            labels[idx] = input_ids[idx]
                            input_ids[idx] = tokenizer.mask_token_id
                            current_mask_group[idx] = 1
                            break

            new_input_ids.append(input_ids)
            new_labels.append(labels)
            mask_group_ids.append(current_mask_group)
            target_rhyme_mask.append(current_target_rhyme)

        tokenized["input_ids"] = new_input_ids
        tokenized["labels"] = new_labels
        tokenized["mask_group_ids"] = mask_group_ids
        tokenized["target_rhyme_mask"] = target_rhyme_mask
        tokenized["is_rhyming"] = is_rhyming_sample
        return tokenized

    tokenized_datasets = filtered_dataset.map(
        process_data,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Processing"
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    dataloader = DataLoader(tokenized_datasets, batch_size=BATCH_SIZE, collate_fn=data_collator)

    head_accumulators = None
    total_rhyme_samples = 0

    for batch in tqdm(dataloader, desc="Scanning Heads"):
        mask_groups = batch.pop("mask_group_ids").to(device)
        target_rhyme = batch.pop("target_rhyme_mask").to(device)
        is_rhyming = batch.pop("is_rhyming").to(device)

        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = model(**batch)
            attentions = outputs.attentions
            all_layers_attn = torch.stack(attentions)

            # Initialize accumulator on first run
            if head_accumulators is None:
                num_layers, _, num_heads, _, _ = all_layers_attn.shape
                head_accumulators = torch.zeros((num_layers, num_heads), device=device)

            # Calculate head importance
            batch_size = mask_groups.shape[0]

            for b in range(batch_size):
                if is_rhyming[b] == 0: continue

                # Get mask indices and rhyme indices
                mask_indices = (mask_groups[b] == 1).nonzero(as_tuple=True)[0]
                rhyme_indices = (target_rhyme[b] == 1).nonzero(as_tuple=True)[0]

                if len(mask_indices) == 0 or len(rhyme_indices) == 0: continue

                # Extract attention sub-matrix
                sample_attn = all_layers_attn[:, b, :, mask_indices, :]

                # Average over mask tokens
                focus_vector = sample_attn.mean(dim=2)

                # Sum attention put on rhyming tokens
                rhyme_mass = focus_vector[:, :, rhyme_indices].sum(dim=-1)

                # Store results
                head_accumulators += rhyme_mass
                total_rhyme_samples += 1

    # Reporting
    print(f"\nHead Analysis Results for {model_id}:")
    print("=" * 60)

    if total_rhyme_samples > 0:
        # Calculate average
        avg_head_attn = head_accumulators / total_rhyme_samples

        # Find best head
        max_val = torch.max(avg_head_attn)
        best_flat_idx = torch.argmax(avg_head_attn).item()
        best_layer = best_flat_idx // avg_head_attn.shape[1]
        best_head = best_flat_idx % avg_head_attn.shape[1]

        print(f"Analyzed {total_rhyme_samples} rhyming samples.")
        print(f"MOST specialized head for rhymes: Layer {best_layer + 1}, Head {best_head + 1}")
        print(f"Average Attention Mass: {max_val:.4f}")

        # Top 3 heads
        print("\nTop 3 Heads:")
        flat_attn = avg_head_attn.flatten()
        top_k = torch.topk(flat_attn, 3)
        for val, idx in zip(top_k.values, top_k.indices):
            l = idx.item() // avg_head_attn.shape[1]
            h = idx.item() % avg_head_attn.shape[1]
            print(f"  - Layer {l+1}, Head {h+1}: {val:.4f}")

    else:
        print("No valid rhyming samples found.")

    print("-" * 60)

    del model
    torch.cuda.empty_cache()